> 数据管线（data pipeline）优于复杂算法

- https://cursor.com/cn/blog/tab-rl
- mdp
    - tab model 作为 policy model，$\pi_\theta(a|s)$（它显然也是一个比较强的文本语言模型，只不过是专门的 coding language model）
    - state：代码上下文及光标位置，
    - action：输出具体的代码补全建议，或者 选择“保持沉默”（不显示建议）。
    - reward：**真实**的用户反馈，即用户是否接受了补全的建议。
        - 文中提到了 reward shaping 的一种策略（权衡“建议接受率”与“打扰用户”），接受(accept)： +0.75, 拒绝(reject)：-0.25，不显示(no show)：0。
- REINFORCE 的 on-policy 性质
    - $J(\theta)=\mathbb E_{s\sim P,a\sim\pi}[R(s,a)]$
    - the gradient: $\nabla_\theta J(\theta)=\mathbb E_{s\sim P(s),a\sim \pi_\theta(a|s)}[\nabla_\theta\log\pi_\theta(a|s)\cdot R(s,a)]$
        -  策略梯度定理 (Policy Gradient Theorem)，梯度的计算依赖于期望 $E_{a \sim \pi_\theta}[\dots]$。
        -  用于更新参数 $\theta$ 的样本（Action）必须是由当前策略 $\pi_\theta$ 采样产生的。
        -  如果使用旧模型产生的数据（Off-Policy），数据分布会发生偏移（Distribution Shift），导致梯度的估计有偏，无法有效指引当前模型的优化。
- On-Policy 的极高时效要求：文中明确指出，On-Policy 算法的核心约束是数据必须新鲜。Cursor 的架构要求从“部署模型”到“收集数据”再到“开始训练”的闭环在 1.5 - 2 小时 内完成。
- 极速闭环 (Loop)：为了满足 On-Policy 的要求，Cursor 构建了一套高频迭代的 Pipeline。
    - Deploy：将当前训练好的 Checkpoint 实时部署给全量或灰度用户。
    - Rollout：用户在实际编码中通过接受/拒绝建议，产生真实的 Trace 数据 $(s, a, r)$。
    - Update：收集这些由当前模型产生的新鲜数据，进行梯度更新。